In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# prob-91

In [0]:
matches_schema = StructType([
    StructField("match_id", IntegerType(), False),
    StructField("winning_team_id", IntegerType(), True),
    StructField("losing_team_id", IntegerType(), True),
    StructField("goals_won", IntegerType(), True)
])
matches_data = [
    (1, 1001, 1007, 1),
    (2, 1007, 1001, 2),
    (3, 1006, 1003, 3),
    (4, 1001, 1003, 1),
    (5, 1007, 1001, 1),
    (6, 1006, 1003, 2),
    (7, 1006, 1001, 3),
    (8, 1007, 1003, 5),
    (9, 1001, 1003, 1),
    (10, 1007, 1006, 2),
    (11, 1006, 1003, 3),
    (12, 1001, 1003, 4),
    (13, 1001, 1006, 2),
    (14, 1007, 1001, 4),
    (15, 1006, 1007, 3),
    (16, 1001, 1003, 3),
    (17, 1001, 1007, 3),
    (18, 1006, 1007, 2),
    (19, 1003, 1001, 1),
    (20, 1001, 1007, 3),
    (21, 1001, 1003, 3)
]



In [0]:
teaminfo_schema = StructType([
    StructField("team_id", IntegerType(), False),
    StructField("team_name", StringType(), True)
])

teaminfo_data = [
    (1001, 'Nickmiesters'),
    (1003, 'sunrisers'),
    (1006, 'Philipines prates'),
    (1007, 'Smashers')
]
matches_df = spark.createDataFrame(matches_data, schema=matches_schema)
teaminfo_df = spark.createDataFrame(teaminfo_data, schema=teaminfo_schema)

In [0]:
teaminfo_df.display()
matches_df.display()

team_id,team_name
1001,Nickmiesters
1003,sunrisers
1006,Philipines prates
1007,Smashers


match_id,winning_team_id,losing_team_id,goals_won
1,1001,1007,1
2,1007,1001,2
3,1006,1003,3
4,1001,1003,1
5,1007,1001,1
6,1006,1003,2
7,1006,1001,3
8,1007,1003,5
9,1001,1003,1
10,1007,1006,2


In [0]:
match_union=matches_df.select(col("winning_team_id").alias("team_id"), lit(1).alias("point"),col("goals_won").alias("goals"))\
        .union(
            matches_df.select(col("losing_team_id").alias("team_id"), lit(-1).alias("point"),lit(0).alias("goals"))
            )
match_union.groupBy("team_id").agg(sum("point").alias("point"),sum(col("goals")).alias("goals"))\
            .orderBy(col("point").desc(),col("goals").desc()).display()

team_id,point,goals
1001,4,21
1006,4,16
1007,0,14
1003,-8,1


# prob -90

In [0]:
department_schema = StructType([
    StructField("dep_id", IntegerType(), False),
    StructField("dep_name", StringType(), True)
])

department_data = [
    (1, 'Electronics'),
    (2, 'Furniture'),
    (3, 'Clothing')
]

department_df = spark.createDataFrame(department_data, schema=department_schema)

In [0]:
empdetails_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("first_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("dep_id", IntegerType(), True)
])

empdetails_data = [
    (101, 'Alice', 'F', 1),
    (102, 'Bob', 'M', 1),
    (103, 'Charlie', 'M', 2),
    (104, 'Diana', 'F', 2),
    (105, 'Ethan', 'M', 3),
    (106, 'Fiona', 'F', 3)
]

empdetails_df = spark.createDataFrame(empdetails_data, schema=empdetails_schema)

In [0]:
client_schema = StructType([
    StructField("client_id", IntegerType(), False),
    StructField("client_name", StringType(), True)
])

client_data = [
    (1, 'Amazon'),
    (2, 'Walmart'),
    (3, 'Costco'),
    (4, 'Target'),
    (5, 'BestBuy')
]

client_df = spark.createDataFrame(client_data, schema=client_schema)

In [0]:
empsales_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("client_id", IntegerType(), False),
    StructField("sales", IntegerType(), True)
])

empsales_data = [
    (101, 1, 5000),
    (101, 2, 3000),
    (102, 1, 7000),
    (102, 3, 2000),
    (103, 2, 4000),
    (103, 4, 3000),
    (104, 4, 6000),
    (105, 5, 8000),
    (106, 3, 5000),
    (106, 5, 2000)
]

empsales_df = spark.createDataFrame(empsales_data, schema=empsales_schema)

In [0]:
emp_join=empdetails_alias("e").join(empsales_dalias("c"), col("e.emp_id") == col("c.emp_id"),"left").select(col("e.*"),col("c.client_id"),col("c.sales"))
emp_join.show()
emp=emp_join.groupBy(col("dep_id"),col("emp_id")).agg(sum("sales").alias("sales")).orderBy(col("sales").desc())
client=emp_join.groupBy(col("dep_id"),col("client_id")).agg(sum("sales").alias("sales")).orderBy(col("sales").desc())
w1=Window.partitionBy(col("dep_id")).orderBy(col("sales").desc())
w2=Window.partitionBy(col("dep_id")).orderBy(col("sales").desc())
emp_rnk=emp.withColumn("e_rnk",dense_rank().over(w1)).filter(col("e_rnk")==1)
cli_rnk=client.withColumn("c_rnk",dense_rank().over(w2)).filter(col("c_rnk")==1)
cli_rnk.alias("client").join(emp_rnk.alias("emp"), col("emp.dep_id") == col("client.dep_id"),"inner").select (col("emp.dep_id"),col("client.client_id"),col("emp.emp_id")).show()

+------+----------+------+------+---------+-----+
|emp_id|first_name|gender|dep_id|client_id|sales|
+------+----------+------+------+---------+-----+
|   101|     Alice|     F|     1|        2| 3000|
|   101|     Alice|     F|     1|        1| 5000|
|   102|       Bob|     M|     1|        3| 2000|
|   102|       Bob|     M|     1|        1| 7000|
|   103|   Charlie|     M|     2|        4| 3000|
|   103|   Charlie|     M|     2|        2| 4000|
|   104|     Diana|     F|     2|        4| 6000|
|   105|     Ethan|     M|     3|        5| 8000|
|   106|     Fiona|     F|     3|        5| 2000|
|   106|     Fiona|     F|     3|        3| 5000|
+------+----------+------+------+---------+-----+

+------+---------+------+
|dep_id|client_id|emp_id|
+------+---------+------+
|     1|        1|   102|
|     2|        4|   103|
|     3|        5|   105|
+------+---------+------+



In [0]:
w1=Window.partitionBy(col("dep_id")).orderBy(col("sales").desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)
emp.withColumn("e_rnk",dense_rank().over(w1)).display()


dep_id,emp_id,sales,e_rnk
1,102,9000,1
1,101,8000,2
2,103,7000,1
2,104,6000,2
3,105,8000,1
3,106,7000,2


# prob-89

In [0]:



# Schema
orders_schema = ''' customer_id int,
                order_date string,
                coupon_code string
'''

# Data
orders_data = [
    (1, '2025-01-10', None),
    (1, '2025-02-05', None),
    (1, '2025-02-20', None),
    (1, '2025-03-01', None),
    (1, '2025-03-10', None),
    (1, '2025-03-15', 'DISC10'),
    (2, '2025-02-02', None),
    (2, '2025-02-05', None),
    (2, '2025-03-05', None),
    (2, '2025-03-18', None),
    (2, '2025-03-20', None),
    (2, '2025-03-22', None),
    (2, '2025-04-02', None),
    (2, '2025-04-10', None),
    (2, '2025-04-15', 'DISC20'),
    (2, '2025-04-16', None),
    (2, '2025-04-18', None),
    (2, '2025-04-20', 'DISC20'),
    (3, '2025-03-05', None),
    (3, '2025-04-10', None),
    (3, '2025-05-15', 'DISC30'),
    (4, '2025-02-01', None),
    (4, '2025-04-05', 'DISC40'),
    (5, '2025-01-03', None),
    (5, '2025-02-05', None),
    (5, '2025-02-15', None),
    (5, '2025-03-01', None),
    (5, '2025-03-08', 'DISC50'),
    (5, '2025-03-20', None),
    (6, '2025-01-05', None),
    (6, '2025-03-02', None),
    (6, '2025-03-15', None),
    (6, '2025-05-05', None),
    (6, '2025-05-10', None),
    (6, '2025-05-25', 'DISC60')
]

# Create DataFrame
orders_df = spark.createDataFrame(orders_data, schema=orders_schema)\
        .withColumn("order_date",to_date("order_date"))
orders_ddisplay()

customer_id,order_date,coupon_code
1,2025-01-10,null
1,2025-02-05,null
1,2025-02-20,null
1,2025-03-01,null
1,2025-03-10,null
1,2025-03-15,DISC10
2,2025-02-02,null
2,2025-02-05,null
2,2025-03-05,null
2,2025-03-18,null


In [0]:
order1=orders_df.withColumn("order_month",to_date(date_trunc("month",col("order_date"))))
order2=order1.withColumn("last_coupon",last("coupon_code").over(Window.partitionBy(col("customer_id")).orderBy(col("order_date")).rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)))\
    .withColumn("o_count",count("*").over(Window.partitionBy(col("customer_id"),col("order_month")).orderBy(col("order_month"))))\
    .withColumn("diff",months_between(col("order_month"),min(col("order_month")).over(Window.partitionBy(col("customer_id")).orderBy(col("order_date")))).cast("int")+lit(1))
order2.filter(~col("last_coupon").isNull()).filter(col("diff").isin(1,2,3))\
    .groupBy(col("customer_id"),col("coupon_code"))\
    .agg(max(when(col("diff") == 1,col("o_count")).otherwise(lit(0))).alias("first_mon"),max(when(col("diff") == 2,col("o_count")).otherwise(lit(0))).alias("second_mon"),max(when(col("diff") == 3,col("o_count")).otherwise(lit(0))).alias("third_mon"))\
    .filter(col("second_mon")==2*col("first_mon"))\
    .filter(col("third_mon")==3*col("first_mon")).show()

+-----------+-----------+---------+----------+---------+
|customer_id|coupon_code|first_mon|second_mon|third_mon|
+-----------+-----------+---------+----------+---------+
|          1|       NULL|        1|         2|        3|
|          2|       NULL|        2|         4|        6|
+-----------+-----------+---------+----------+---------+



In [0]:

order2.show(6)
pivot_df = order2.groupBy("customer_id") \
    .pivot("diff", [1, 2, 3]) \
    .agg(max("o_count"),sum("o_count")).show()


+-----------+----------+-----------+-----------+-----------+-------+----+
|customer_id|order_date|coupon_code|order_month|last_coupon|o_count|diff|
+-----------+----------+-----------+-----------+-----------+-------+----+
|          1|2025-01-10|       NULL| 2025-01-01|     DISC10|      1|   1|
|          1|2025-02-05|       NULL| 2025-02-01|     DISC10|      2|   2|
|          1|2025-02-20|       NULL| 2025-02-01|     DISC10|      2|   2|
|          1|2025-03-01|       NULL| 2025-03-01|     DISC10|      3|   3|
|          1|2025-03-10|       NULL| 2025-03-01|     DISC10|      3|   3|
|          1|2025-03-15|     DISC10| 2025-03-01|     DISC10|      3|   3|
+-----------+----------+-----------+-----------+-----------+-------+----+
only showing top 6 rows
+-----------+--------------+--------------+--------------+--------------+--------------+--------------+
|customer_id|1_max(o_count)|1_sum(o_count)|2_max(o_count)|2_sum(o_count)|3_max(o_count)|3_sum(o_count)|
+-----------+--------------+

# prob-80

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime

# Initialize Spark
spark = SparkSession.builder.appName("DateTimeExample").getOrCreate()

# Define schema with TimestampType
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

# Sample data with datetime strings
data = [
    (1, datetime.strptime('2025-01-01 10:00:01', '%Y-%m-%d %H:%M:%S')),
    (2, datetime.strptime('2025-01-02 12:30:45', '%Y-%m-%d %H:%M:%S')),
    (3, datetime.strptime('2025-01-03 18:15:00', '%Y-%m-%d %H:%M:%S'))
]


# Create DataFrame
df1 = spark.createDataFrame(data, schema)

# Show DataFrame
df1.show(truncate=False)
df1.printSchema()


+---+-------------------+
|id |event_time         |
+---+-------------------+
|1  |2025-01-01 10:00:01|
|2  |2025-01-02 12:30:45|
|3  |2025-01-03 18:15:00|
+---+-------------------+

root
 |-- id: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [0]:



# Initialize Spark
# spark = SparkSession.builder.appName("TransactionsExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("amount", IntegerType(), True),
    StructField("tran_Date", StringType(), True)
])

# Data (same as your SQL inserts)
data = [
    (1, 101, 500, '2025-01-01 10:00:01'),
    (2, 201, 500, '2025-01-01 10:00:01'),
    (3, 102, 300, '2025-01-02 00:50:01'),
    (4, 202, 300, '2025-01-02 00:50:01'),
    (5, 101, 700, '2025-01-03 06:00:01'),
    (6, 202, 700, '2025-01-03 06:00:01'),
    (7, 103, 200, '2025-01-04 03:00:01'),
    (8, 203, 200, '2025-01-04 03:00:01'),
    (9, 101, 400, '2025-01-05 00:10:01'),
    (10, 201, 400, '2025-01-05 00:10:01'),
    (11, 101, 500, '2025-01-07 10:10:01'),
    (12, 201, 500, '2025-01-07 10:10:01'),
    (13, 102, 200, '2025-01-03 10:50:01'),
    (14, 202, 200, '2025-01-03 10:50:01'),
    (15, 103, 500, '2025-01-01 11:00:01'),
    (16, 101, 500, '2025-01-01 11:00:01'),
    (17, 203, 200, '2025-11-01 11:00:01'),
    (18, 201, 200, '2025-11-01 11:00:01')
]

# Create DataFrame
df = spark.createDataFrame(data, schema)
dwithColumn("tran_Date",to_timestamp("tran_Date",'yyyy-MM-dd HH:mm:ss'))
# Show DataFrame
dshow(truncate=False)
dprintSchema()



+--------------+-----------+------+-------------------+
|transaction_id|customer_id|amount|tran_Date          |
+--------------+-----------+------+-------------------+
|1             |101        |500   |2025-01-01 10:00:01|
|2             |201        |500   |2025-01-01 10:00:01|
|3             |102        |300   |2025-01-02 00:50:01|
|4             |202        |300   |2025-01-02 00:50:01|
|5             |101        |700   |2025-01-03 06:00:01|
|6             |202        |700   |2025-01-03 06:00:01|
|7             |103        |200   |2025-01-04 03:00:01|
|8             |203        |200   |2025-01-04 03:00:01|
|9             |101        |400   |2025-01-05 00:10:01|
|10            |201        |400   |2025-01-05 00:10:01|
|11            |101        |500   |2025-01-07 10:10:01|
|12            |201        |500   |2025-01-07 10:10:01|
|13            |102        |200   |2025-01-03 10:50:01|
|14            |202        |200   |2025-01-03 10:50:01|
|15            |103        |500   |2025-01-01 11

In [0]:
df1=dwithColumn('buyer',lead(col('customer_id'),1).over(Window.orderBy(col('transaction_id')))) \
    .withColumnRenamed('customer_id','seller')\
    .filter(col("transaction_id")%2!=0)\
    .groupBy('buyer','seller').agg(count('*').alias('total_count'))

seller_list = [row.seller for row in df1.select("seller").distinct().collect()]
seller_list11 = df1.select("seller").distinct().collect()[0]
print(seller_list11[0])
buyer_list = [row.buyer for row in df1.select("buyer").distinct().collect()]

df1.filter(~col("seller").isin(buyer_list)).filter(~col("buyer").isin(seller_list)).show()       
print(seller_list)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


101
+-----+------+-----------+
|buyer|seller|total_count|
+-----+------+-----------+
|  202|   102|          2|
+-----+------+-----------+

[101, 102, 103, 203]


# prob-79

In [0]:

# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("experience", IntegerType(), True),
    StructField("sql", IntegerType(), True),
    StructField("algo", IntegerType(), True),
    StructField("bug_fixing", IntegerType(), True)
])

# Data (same as your SQL inserts, with None for NULL)
data = [
    (1, 3, 100, None, 50),
    (2, 5, None, 100, 100),
    (3, 1, 100, 100, 100),
    (4, 5, 100, 50, None),
    (5, 5, 100, 100, 100)
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Show DataFrame
dshow()
dprintSchema()


+---+----------+----+----+----------+
| id|experience| sql|algo|bug_fixing|
+---+----------+----+----+----------+
|  1|         3| 100|NULL|        50|
|  2|         5|NULL| 100|       100|
|  3|         1| 100| 100|       100|
|  4|         5| 100|  50|      NULL|
|  5|         5| 100| 100|       100|
+---+----------+----+----+----------+

root
 |-- id: integer (nullable = true)
 |-- experience: integer (nullable = true)
 |-- sql: integer (nullable = true)
 |-- algo: integer (nullable = true)
 |-- bug_fixing: integer (nullable = true)



In [0]:
dwithColumn('best_score', when(
    ((col('sql').isNull()) | (col('sql')==100) ) &
    ((col('algo').isNull()) | (col('algo')==100) ) &
    ((col('bug_fixing').isNull()) | (col('bug_fixing')==100) ),1
).otherwise(0)).groupBy("experience").agg(count('*').alias("total_user"),sum('best_score').alias("best_score")).show()

+----------+----------+----------+
|experience|total_user|best_score|
+----------+----------+----------+
|         3|         1|         0|
|         5|         3|         2|
|         1|         1|         1|
+----------+----------+----------+



# prob-78

In [0]:

schema = StructType([
    StructField("matchid", IntegerType(), True),
    StructField("ballnumber", IntegerType(), True),
    StructField("inningno", IntegerType(), True),
    StructField("overs", FloatType(), True),
    StructField("outcome", StringType(), True),
    StructField("batter", StringType(), True),
    StructField("bowler", StringType(), True),
    StructField("score", IntegerType(), True)  # Changed to IntegerType
])


data = [
    (1,1,1,0.1,'0','Mohammed Shami','Devon Conway',0),
    (1,2,1,0.2,'1lb','Mohammed Shami','Devon Conway',1),
    (1,3,1,0.3,'0','Mohammed Shami','Ruturaj Gaikwad',0),
    (1,4,1,0.4,'1','Mohammed Shami','Ruturaj Gaikwad',1),
    (1,5,1,0.5,'0','Mohammed Shami','Devon Conway',0),
    (1,6,1,0.6,'0','Mohammed Shami','Devon Conway',0),
    (1,7,1,1.1,'4','Hardik Pandya','Ruturaj Gaikwad',4),
    (1,8,1,1.2,'0','Hardik Pandya','Ruturaj Gaikwad',0),
    (1,9,1,1.3,'4','Hardik Pandya','Ruturaj Gaikwad',4),
    (1,10,1,1.4,'1','Hardik Pandya','Ruturaj Gaikwad',1),
    (1,11,1,1.5,'1','Hardik Pandya','Devon Conway',1),
    (1,12,1,1.6,'1','Hardik Pandya','Ruturaj Gaikwad',1),
    (1,13,2,2.1,'1','Ruturaj Gaikwad','Mohammed Shami',1),
    (1,14,2,2.2,'w','Devon Conway','Mohammed Shami',0),
    (1,15,2,2.3,'0','Moeen Ali','Mohammed Shami',0),
    (1,16,2,2.4,'0','Moeen Ali','Mohammed Shami',0),
    (1,17,2,2.5,'0','Moeen Ali','Mohammed Shami',0),
    (1,18,2,2.6,'0','Moeen Ali','Mohammed Shami',0),
    (1,19,2,3.1,'6','Ruturaj Gaikwad','Josh Little',6),
    (1,20,2,3.2,'4','Ruturaj Gaikwad','Josh Little',4),
    (1,21,2,3.3,'1','Ruturaj Gaikwad','Josh Little',1),
    (1,22,2,3.4,'0','Moeen Ali','Josh Little',0),
    (1,23,2,3.5,'4','Moeen Ali','Josh Little',4),
    (1,24,2,3.6,'0','Moeen Ali','Josh Little',0),

    (2,1,1,4.1,'0','Mohammed Shami','Ruturaj Gaikwad',0),
    (2,2,1,4.2,'1','Mohammed Shami','Ruturaj Gaikwad',1),
    (2,3,1,4.3,'4','Mohammed Shami','Moeen Ali',4),
    (2,4,1,4.4,'1nb','Mohammed Shami','Moeen Ali',1),
    (2,5,1,4.4,'6','Mohammed Shami','Moeen Ali',6),
    (2,6,1,4.5,'4','Mohammed Shami','Moeen Ali',4),
    (2,7,1,4.6,'1','Mohammed Shami','Moeen Ali',1),
    (2,8,1,5.1,'0','Rashid Khan','Moeen Ali',0),
    (2,9,1,5.2,'0','Rashid Khan','Moeen Ali',0),
    (2,10,1,5.3,'0','Rashid Khan','Moeen Ali',0),
    (2,11,1,5.4,'4','Rashid Khan','Moeen Ali',4),
    (2,12,1,5.5,'w','Rashid Khan','Moeen Ali',0),
    (2,13,1,5.6,'1','Rashid Khan','Ben S',1),
    (2,14,2,6.1,'0','Ben S','Hardik Pandya',0),
    (2,15,2,6.2,'1','Ben S','Hardik Pandya',1),
    (2,16,2,6.3,'6','Ruturaj Gaikwad','Hardik Pandya',6),
    (2,17,2,6.4,'6','Ruturaj Gaikwad','Hardik Pandya',6),
    (2,18,2,6.5,'0','Ruturaj Gaikwad','Hardik Pandya',0),
    (2,19,2,6.6,'0','Ruturaj Gaikwad','Hardik Pandya',0),
    (2,20,2,7.1,'1','Ben S','Rashid Khan',1),
    (2,21,2,7.2,'1','Ruturaj Gaikwad','Rashid Khan',1),
    (2,22,2,7.3,'4','Ben S','Rashid Khan',4),
    (2,23,2,7.4,'w','Ben S','Rashid Khan',0),
    (2,24,2,7.5,'1','Ambati Rayudu','Rashid Khan',1),
    (2,25,2,7.6,'1','Ruturaj Gaikwad','Rashid Khan',1),

    (3,1,1,8.1,'6','Alzarri Joseph','Ruturaj Gaikwad',6),
    (3,2,1,8.2,'0','Alzarri Joseph','Ruturaj Gaikwad',0),
    (3,3,1,8.3,'0','Alzarri Joseph','Ruturaj Gaikwad',0),
    (3,4,1,8.4,'6','Alzarri Joseph','Ruturaj Gaikwad',6),
    (3,5,1,8.5,'0','Alzarri Joseph','Ruturaj Gaikwad',0),
    (3,6,1,8.6,'6','Alzarri Joseph','Ruturaj Gaikwad',6),
    (3,7,1,9.1,'0','Rashid Khan','Brett Lee',0),
    (3,8,1,9.2,'1','Rashid Khan','Brett Lee',1),
    (3,9,1,9.3,'1','Rashid Khan','Ruturaj Gaikwad',1),
    (3,10,1,9.4,'1','Rashid Khan','Brett Lee',1),
    (3,11,1,9.5,'0','Rashid Khan','Ruturaj Gaikwad',0),
    (3,12,1,9.6,'0','Rashid Khan','Ruturaj Gaikwad',0),
    (3,13,2,10.1,'0','Ambati Rayudu','Josh Little',0),
    (3,14,2,10.2,'0','Ambati Rayudu','Josh Little',0),
    (3,15,2,10.3,'0','Ambati Rayudu','Josh Little',0),
    (3,16,2,10.4,'1','Ambati Rayudu','Josh Little',1),
    (3,17,2,10.5,'0','Ruturaj Gaikwad','Josh Little',0),
    (3,18,2,10.6,'6','Ruturaj Gaikwad','Josh Little',6),
    (3,19,2,11.1,'1','Ambati Rayudu','Yash Dayal',1),
    (3,20,2,11.2,'0','Ruturaj Gaikwad','Yash Dayal',0),
    (3,21,2,11.3,'6','Ruturaj Gaikwad','Yash Dayal',6),
    (3,22,2,11.4,'0','Ruturaj Gaikwad','Yash Dayal',0),
    (3,23,2,11.5,'1','Ruturaj Gaikwad','Yash Dayal',1),
    (3,24,2,11.6,'6','Ambati Rayudu','Yash Dayal',6),

    (4,1,1,12.1,'1','Josh Little','Ruturaj Gaikwad',1),
    (4,2,1,12.2,'1','Josh Little','Brett Lee',1),
    (4,3,1,12.3,'4','Josh Little','Ruturaj Gaikwad',4),
    (4,4,1,12.4,'1','Josh Little','Ruturaj Gaikwad',1),
    (4,5,1,12.5,'w','Josh Little','Brett Lee',0),
    (4,6,1,12.6,'0','Josh Little','Shivam Dube',0),
    (4,7,1,13.1,'1','Alzarri Joseph','Ruturaj Gaikwad',1),
    (4,8,1,13.2,'1','Alzarri Joseph','Shivam Dube',1),
    (4,9,1,13.3,'1','Alzarri Joseph','Ruturaj Gaikwad',1),
    (4,10,1,13.4,'0','Alzarri Joseph','Shivam Dube',0),
    (4,11,1,13.5,'0','Alzarri Joseph','Shivam Dube',0),
    (4,12,1,13.6,'1','Alzarri Joseph','Shivam Dube',1),
    (4,13,2,14.1,'1','Shivam Dube','Hardik Pandya',1),
    (4,14,2,14.2,'1','Ruturaj Gaikwad','Hardik Pandya',1),
    (4,15,2,14.3,'4lb','Shivam Dube','Hardik Pandya',4),
    (4,16,2,14.4,'1','Shivam Dube','Hardik Pandya',1),
    (4,17,2,14.5,'0','Ruturaj Gaikwad','Hardik Pandya',0),
    (4,18,2,14.6,'1','Ruturaj Gaikwad','Hardik Pandya',1),
    (4,19,2,15.1,'1','Ruturaj Gaikwad','Alzarri Joseph',1),
    (4,20,2,15.2,'0','Shivam Dube','Alzarri Joseph',0),
    (4,21,2,15.3,'1','Shivam Dube','Alzarri Joseph',1),
    (4,22,2,15.4,'1','Ruturaj Gaikwad','Alzarri Joseph',1),
    (4,23,2,15.5,'2','Shivam Dube','Alzarri Joseph',2),
    (4,24,2,15.6,'2','Shivam Dube','Alzarri Joseph',2)
]

df = spark.createDataFrame(data, schema)

# Show DataFrame
dshow(truncate=False)
dprintSchema()


+-------+----------+--------+-----+-------+---------------+---------------+-----+
|matchid|ballnumber|inningno|overs|outcome|batter         |bowler         |score|
+-------+----------+--------+-----+-------+---------------+---------------+-----+
|1      |1         |1       |0.1  |0      |Mohammed Shami |Devon Conway   |0    |
|1      |2         |1       |0.2  |1lb    |Mohammed Shami |Devon Conway   |1    |
|1      |3         |1       |0.3  |0      |Mohammed Shami |Ruturaj Gaikwad|0    |
|1      |4         |1       |0.4  |1      |Mohammed Shami |Ruturaj Gaikwad|1    |
|1      |5         |1       |0.5  |0      |Mohammed Shami |Devon Conway   |0    |
|1      |6         |1       |0.6  |0      |Mohammed Shami |Devon Conway   |0    |
|1      |7         |1       |1.1  |4      |Hardik Pandya  |Ruturaj Gaikwad|4    |
|1      |8         |1       |1.2  |0      |Hardik Pandya  |Ruturaj Gaikwad|0    |
|1      |9         |1       |1.3  |4      |Hardik Pandya  |Ruturaj Gaikwad|4    |
|1      |10     

In [0]:
batter= dselect(col('batter').alias('player'),col('matchid').alias('batter_matchid'),lit(None).alias('bowler_matchid'))
bowler= dselect(col('bowler').alias('player'),lit(None).alias('batter_matchid'),col('matchid').alias('bowler_matchid'))
df1=batter.union(bowler).distinct()
df1.groupBy('player').agg(countDistinct(coalesce('batter_matchid','bowler_matchid')).alias("total_match"),count('batter_matchid').alias("batter_match"),count('bowler_matchid').alias("bowler_match")).show()


+---------------+-----------+------------+------------+
|         player|total_match|batter_match|bowler_match|
+---------------+-----------+------------+------------+
| Mohammed Shami|          2|           2|           1|
|  Hardik Pandya|          3|           1|           2|
|Ruturaj Gaikwad|          4|           4|           4|
|   Devon Conway|          1|           1|           1|
|      Moeen Ali|          2|           1|           1|
|    Rashid Khan|          2|           2|           1|
|          Ben S|          1|           1|           1|
|  Ambati Rayudu|          2|           2|           0|
| Alzarri Joseph|          2|           2|           1|
|    Josh Little|          3|           1|           2|
|    Shivam Dube|          1|           1|           1|
|      Brett Lee|          2|           0|           2|
|     Yash Dayal|          1|           0|           1|
+---------------+-----------+------------+------------+



# prob-77

In [0]:

# Define schema
schema = StructType([
    StructField("userid", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", StringType(), True)
])

# Data (replace SQL datetime with strings, PySpark will convert to timestamp)
data = [
    (1, 'click', '2023-09-10 09:00:00'),
    (1, 'click', '2023-09-10 10:00:00'),
    (1, 'scroll', '2023-09-10 10:20:00'),
    (1, 'click', '2023-09-10 10:50:00'),
    (1, 'scroll', '2023-09-10 11:40:00'),
    (1, 'click', '2023-09-10 12:40:00'),
    (1, 'scroll', '2023-09-10 12:50:00'),
    (2, 'click', '2023-09-10 09:00:00'),
    (2, 'scroll', '2023-09-10 09:20:00'),
    (2, 'click', '2023-09-10 10:30:00')
]

# Create DataFrame
df = spark.createDataFrame(data, schema)
df=dwithColumn("event_time", to_timestamp(col("event_time")))
# Show DataFrame
dshow(truncate=False)
dprintSchema()


+------+----------+-------------------+
|userid|event_type|event_time         |
+------+----------+-------------------+
|1     |click     |2023-09-10 09:00:00|
|1     |click     |2023-09-10 10:00:00|
|1     |scroll    |2023-09-10 10:20:00|
|1     |click     |2023-09-10 10:50:00|
|1     |scroll    |2023-09-10 11:40:00|
|1     |click     |2023-09-10 12:40:00|
|1     |scroll    |2023-09-10 12:50:00|
|2     |click     |2023-09-10 09:00:00|
|2     |scroll    |2023-09-10 09:20:00|
|2     |click     |2023-09-10 10:30:00|
+------+----------+-------------------+

root
 |-- userid: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [0]:
df1=dwithColumn('previous_time',lag(col('event_time'),1,col('event_time')).over(Window.partitionBy('userid').orderBy('event_time'))).withColumn(
    "diff",
    (unix_timestamp("event_time") - unix_timestamp("previous_time")) / 60
).withColumn('flag',when(col('diff')>30,1).otherwise(0))

df1.withColumn('flag',sum(col('flag')).over(Window.partitionBy('userid').orderBy('event_time'))).groupBy("userid","flag").agg(count('*').alias("count_event"),min("event_time").alias('min_time'),max('event_time').alias('max_time')).withColumn('duration',((unix_timestamp('max_time')-unix_timestamp('min_time'))/60).cast('int')).show(truncate=False)


+------+----+-----------+-------------------+-------------------+--------+
|userid|flag|count_event|min_time           |max_time           |duration|
+------+----+-----------+-------------------+-------------------+--------+
|1     |0   |1          |2023-09-10 09:00:00|2023-09-10 09:00:00|0       |
|1     |1   |3          |2023-09-10 10:00:00|2023-09-10 10:50:00|50      |
|1     |2   |1          |2023-09-10 11:40:00|2023-09-10 11:40:00|0       |
|1     |3   |2          |2023-09-10 12:40:00|2023-09-10 12:50:00|10      |
|2     |0   |2          |2023-09-10 09:00:00|2023-09-10 09:20:00|20      |
|2     |1   |1          |2023-09-10 10:30:00|2023-09-10 10:30:00|0       |
+------+----+-----------+-------------------+-------------------+--------+



In [0]:
import datetime

df = spark.createDataFrame(
    [
        (datetime.datetime(2025, 4, 1, 10, 0, 7), datetime.datetime(2025, 4, 2, 9, 0, 7))
    ],
    ['ts1', 'ts2']
)

dselect(
    '*',
    timestamp_diff('hour', 'ts1', 'ts2')
).show()

df = spark.createDataFrame(
    [
        (datetime.date(2023, 4, 11), datetime.date(2024, 4, 2))
    ],
    ['ts1', 'ts2']
)

dselect(
    '*',
    timestamp_diff('month', 'ts1', 'ts2')
).show()

+-------------------+-------------------+-----------------------------+
|                ts1|                ts2|timestampdiff(hour, ts1, ts2)|
+-------------------+-------------------+-----------------------------+
|2025-04-01 10:00:07|2025-04-02 09:00:07|                           23|
+-------------------+-------------------+-----------------------------+

+----------+----------+------------------------------+
|       ts1|       ts2|timestampdiff(month, ts1, ts2)|
+----------+----------+------------------------------+
|2023-04-11|2024-04-02|                            11|
+----------+----------+------------------------------+



## prob-76

In [0]:


# Initialize Spark
# spark = SparkSession.builder.appName("PhoneNumbersExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("num", StringType(), True)
])

# Data
data = [
    ('1234567780'),
    ('2234578996'),
    ('+1-12244567780'),
    ('+32-2233567889'),
    ('+2-23456987312'),
    ('+91-9087654123'),
    ('+23-9085761324'),
    ('+11-8091013345')
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Show DataFrame
df.show(truncate=False)
df.printSchema()


+--------------+
|num           |
+--------------+
|1234567780    |
|2234578996    |
|+1-12244567780|
|+32-2233567889|
|+2-23456987312|
|+91-9087654123|
|+23-9085761324|
|+11-8091013345|
+--------------+

root
 |-- num: string (nullable = true)



In [0]:
df_position = df.withColumn(
    "at_position",
    instr(col("num"), "-")
).show()

df_mob = df.withColumn(
    "mob",
    get(split(col("num"), "-"),1)
).show()




# UDF to check unique digits
def has_unique_digits(phone):
    digits = ''.join(ch for ch in phone if ch.isdigit())  # Remove non-digits
    return len(digits) == len(set(digits))

unique_digits_udf = udf(has_unique_digits, BooleanType())


df1=df.withColumn("new_num",when( instr(col("num"),'-')==0,col('num')).otherwise(get(split(col("num"),"-"),1)))
df1.show()

# Apply UDF
df_filtered = df1.filter(unique_digits_udf(col("new_num")))

df_filtered.show(truncate=False)



+--------------+-----------+
|           num|at_position|
+--------------+-----------+
|    1234567780|          0|
|    2234578996|          0|
|+1-12244567780|          3|
|+32-2233567889|          4|
|+2-23456987312|          3|
|+91-9087654123|          4|
|+23-9085761324|          4|
|+11-8091013345|          4|
+--------------+-----------+

+--------------+-----------+
|           num|        mob|
+--------------+-----------+
|    1234567780|       NULL|
|    2234578996|       NULL|
|+1-12244567780|12244567780|
|+32-2233567889| 2233567889|
|+2-23456987312|23456987312|
|+91-9087654123| 9087654123|
|+23-9085761324| 9085761324|
|+11-8091013345| 8091013345|
+--------------+-----------+

+--------------+-----------+
|           num|    new_num|
+--------------+-----------+
|    1234567780| 1234567780|
|    2234578996| 2234578996|
|+1-12244567780|12244567780|
|+32-2233567889| 2233567889|
|+2-23456987312|23456987312|
|+91-9087654123| 9087654123|
|+23-9085761324| 9085761324|
|+11-8091013

## ### prob-75

In [0]:

from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

# Initialize Spark
spark = SparkSession.builder.appName("PollsExample").getOrCreate()

# Define schema for polls
polls_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("poll_id", StringType(), True),
    StructField("poll_option_id", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("created_date", StringType(), True)
])

# Data for polls
polls_data = [
    ('id1', 'p1', 'A', 200, '2021-12-01'),
    ('id2', 'p1', 'C', 250, '2021-12-01'),
    ('id3', 'p1', 'A', 200, '2021-12-01'),
    ('id4', 'p1', 'B', 500, '2021-12-01'),
    ('id5', 'p1', 'C', 50, '2021-12-01'),
    ('id6', 'p1', 'D', 500, '2021-12-01'),
    ('id7', 'p1', 'C', 200, '2021-12-01'),
    ('id8', 'p1', 'A', 100, '2021-12-01'),
    ('id9', 'p2', 'A', 300, '2023-01-10'),
    ('id10', 'p2', 'C', 400, '2023-01-11'),
    ('id11', 'p2', 'B', 250, '2023-01-12'),
    ('id12', 'p2', 'D', 600, '2023-01-13'),
    ('id13', 'p2', 'C', 150, '2023-01-14'),
    ('id14', 'p2', 'A', 100, '2023-01-15'),
    ('id15', 'p2', 'C', 200, '2023-01-16')
]

# Create polls DataFrame
polls_df1 = spark.createDataFrame(polls_data, polls_schema)

# Define schema for poll_answers
answers_schema = StructType([
    StructField("poll_id", StringType(), True),
    StructField("correct_option_id", StringType(), True)
])

# Data for poll_answers
answers_data = [
    ('p1', 'C'),
    ('p2', 'A')
]

# Create poll_answers DataFrame
answers_df = spark.createDataFrame(answers_data, answers_schema)

# Show DataFrames
print("Polls DataFrame:")
polls_df=polls_df1.withColumn('created_date',to_date('created_date','yyyy-MM-dd'))
polls_dshow(truncate=False)
polls_dprintSchema()
print("Poll Answers DataFrame:")
answers_dshow(truncate=False)


Polls DataFrame:
+-------+-------+--------------+------+------------+
|user_id|poll_id|poll_option_id|amount|created_date|
+-------+-------+--------------+------+------------+
|id1    |p1     |A             |200   |2021-12-01  |
|id2    |p1     |C             |250   |2021-12-01  |
|id3    |p1     |A             |200   |2021-12-01  |
|id4    |p1     |B             |500   |2021-12-01  |
|id5    |p1     |C             |50    |2021-12-01  |
|id6    |p1     |D             |500   |2021-12-01  |
|id7    |p1     |C             |200   |2021-12-01  |
|id8    |p1     |A             |100   |2021-12-01  |
|id9    |p2     |A             |300   |2023-01-10  |
|id10   |p2     |C             |400   |2023-01-11  |
|id11   |p2     |B             |250   |2023-01-12  |
|id12   |p2     |D             |600   |2023-01-13  |
|id13   |p2     |C             |150   |2023-01-14  |
|id14   |p2     |A             |100   |2023-01-15  |
|id15   |p2     |C             |200   |2023-01-16  |
+-------+-------+------------

In [0]:
df2 = polls_dalias('p').join(
    answers_dalias('a'),
    col("p.poll_id") == col("a.poll_id"),
    'inner'
).select(
    col("p.*"),
    col("a.correct_option_id")
).withColumn(
    "status",
    when(col("correct_option_id") == col("poll_option_id"), "winner").otherwise("looser")
)

winner = df2.filter(col("status") == 'winner') \
    .withColumn("winner_amt", sum("amount").over(Window.partitionBy("poll_id"))).withColumn("proportion",col("amount")/col("winner_amt"))

# display(winner)

looser = df2.filter(col("status") == 'looser') \
    .groupBy("poll_id").agg(sum("amount").alias("looser_amt"))

# display(looser)
winner.join(looser, on='poll_id').select(winner['*'], looser['looser_amt']).withColumn("final_settle",col("looser_amt")*col("proportion")).show()

+-------+-------+--------------+------+------------+-----------------+------+----------+----------+----------+------------+
|user_id|poll_id|poll_option_id|amount|created_date|correct_option_id|status|winner_amt|proportion|looser_amt|final_settle|
+-------+-------+--------------+------+------------+-----------------+------+----------+----------+----------+------------+
|    id7|     p1|             C|   200|  2021-12-01|                C|winner|       500|       0.4|      1500|       600.0|
|   id14|     p2|             A|   100|  2023-01-15|                A|winner|       400|      0.25|      1600|       400.0|
|    id5|     p1|             C|    50|  2021-12-01|                C|winner|       500|       0.1|      1500|       150.0|
|    id9|     p2|             A|   300|  2023-01-10|                A|winner|       400|      0.75|      1600|      1200.0|
|    id2|     p1|             C|   250|  2021-12-01|                C|winner|       500|       0.5|      1500|       750.0|
+-------

# prob-74
#

In [0]:

from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, IntegerType

# Initialize Spark
spark = SparkSession.builder.appName("NumbersExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("n", IntegerType(), True)
])

# Data
data = [(1,), (2,), (3,), (4,), (5,), (9,)]

# Create DataFrame
numbers_df = spark.createDataFrame(data, schema)

# Show DataFrame
numbers_df.show()


+---+
|  n|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
|  9|
+---+



In [0]:

# Step 1: Get min and max
min_val = numbers_dagg(min("n")).collect()[0][0]
max_val = numbers_dagg({"n": "max"}).collect()[0][0]

# Step 2: Generate full range DataFrame
full_range_df = spark.range(min_val, max_val + 1).toDF("num")

# Step 3: Find missing numbers using left anti join
missing_df = full_range_df.join(numbers_df, full_range_df.num == numbers_df.n, "left_anti")

missing_dshow()


+---+
|num|
+---+
|  6|
|  7|
|  8|
+---+



In [0]:
numbers_dalias('n1').join(numbers_dalias('n2'), col('n1.n') <= col('n2.n'), "inner").select("n1.n","n2.n").show()

+---+---+
|  n|  n|
+---+---+
|  1|  1|
|  1|  2|
|  1|  3|
|  1|  4|
|  1|  5|
|  1|  9|
|  2|  2|
|  2|  3|
|  2|  4|
|  2|  5|
|  2|  9|
|  3|  3|
|  3|  4|
|  3|  5|
|  3|  9|
|  4|  4|
|  4|  5|
|  4|  9|
|  5|  5|
|  5|  9|
+---+---+
only showing top 20 rows


In [0]:
numbers_dcreateOrReplaceTempView("numbers")

number_df1 = spark.sql("""
with recursive cte as (
    select n, 1 as total from numbers
    union all
    select cte.n, total + 1 from cte where total + 1 <= cte.n
)
select * from cte order by n
""")
display(number_df1)

n,total
1,1
2,1
2,2
3,1
3,2
3,3
4,1
4,2
4,3
4,4


In [0]:
display(
    numbers_dwithColumn(
        "total",
        explode(
            sequence(
                lit("1"),
                col("n")
            )
        )
    )
)

n,total
1,1
2,1
2,2
3,1
3,2
3,3
4,1
4,2
4,3
4,4


## prob-73

In [0]:

from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

# Initialize Spark
spark = SparkSession.builder.appName("SubscriptionHistoryExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("marketplace", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("event", StringType(), True),
    StructField("subscription_period", IntegerType(), True)
])

# Data (replace NULL with None)
data = [
    (1, 'India', '2020-01-05', 'S', 6),
    (1, 'India', '2020-12-05', 'R', 1),
    (1, 'India', '2021-02-05', 'C', None),
    (2, 'India', '2020-02-15', 'S', 12),
    (2, 'India', '2020-11-20', 'C', None),
    (3, 'USA', '2019-12-01', 'S', 12),
    (3, 'USA', '2020-12-01', 'R', 12),
    (4, 'USA', '2020-01-10', 'S', 6),
    (4, 'USA', '2020-09-10', 'R', 3),
    (4, 'USA', '2020-12-25', 'C', None),
    (5, 'UK', '2020-06-20', 'S', 12),
    (5, 'UK', '2020-11-20', 'C', None),
    (6, 'UK', '2020-07-05', 'S', 6),
    (6, 'UK', '2021-03-05', 'R', 6),
    (7, 'Canada', '2020-08-15', 'S', 12),
    (8, 'Canada', '2020-09-10', 'S', 12),
    (8, 'Canada', '2020-12-10', 'C', None),
    (9, 'Canada', '2020-11-10', 'S', 1)
]

# Create DataFrame
df_create= spark.createDataFrame(data, schema)
df=df_create.withColumn("event_date", to_date(col("event_date"), "yyyy-MM-dd"))
# Show DataFrame
dshow(truncate=False)
dprintSchema()



+-----------+-----------+----------+-----+-------------------+
|customer_id|marketplace|event_date|event|subscription_period|
+-----------+-----------+----------+-----+-------------------+
|1          |India      |2020-01-05|S    |6                  |
|1          |India      |2020-12-05|R    |1                  |
|1          |India      |2021-02-05|C    |NULL               |
|2          |India      |2020-02-15|S    |12                 |
|2          |India      |2020-11-20|C    |NULL               |
|3          |USA        |2019-12-01|S    |12                 |
|3          |USA        |2020-12-01|R    |12                 |
|4          |USA        |2020-01-10|S    |6                  |
|4          |USA        |2020-09-10|R    |3                  |
|4          |USA        |2020-12-25|C    |NULL               |
|5          |UK         |2020-06-20|S    |12                 |
|5          |UK         |2020-11-20|C    |NULL               |
|6          |UK         |2020-07-05|S    |6            

In [0]:
dfilter(col('event_date')<=lit('2020-12-31').cast("date"))\
    .withColumn("rnk", row_number().over(Window.partitionBy("customer_id").orderBy(col("event_date").desc()))).filter((col("rnk")==1) & (col("event")!='C')).drop("rnk").withColumn("subscription_date",add_months(col('event_date'),col("subscription_period"))).filter(col('subscription_date')>=lit('2020-12-31').cast("date")).show(truncate=False)

+-----------+-----------+----------+-----+-------------------+-----------------+
|customer_id|marketplace|event_date|event|subscription_period|subscription_date|
+-----------+-----------+----------+-----+-------------------+-----------------+
|1          |India      |2020-12-05|R    |1                  |2021-01-05       |
|3          |USA        |2020-12-01|R    |12                 |2021-12-01       |
|6          |UK         |2020-07-05|S    |6                  |2021-01-05       |
|7          |Canada     |2020-08-15|S    |12                 |2021-08-15       |
+-----------+-----------+----------+-----+-------------------+-----------------+



## prob-72

In [0]:

from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

# Initialize Spark
spark = SparkSession.builder.appName("SwipeExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("activity_type", StringType(), True),
    StructField("activity_time", StringType(), True)
])

# Data
data = [
    (1, 'login', '2024-07-23 08:00:00'),
    (1, 'logout', '2024-07-23 12:00:00'),
    (1, 'login', '2024-07-23 13:00:00'),
    (1, 'logout', '2024-07-23 17:00:00'),
    (2, 'login', '2024-07-23 09:00:00'),
    (2, 'logout', '2024-07-23 11:00:00'),
    (2, 'login', '2024-07-23 12:00:00'),
    (2, 'logout', '2024-07-23 15:00:00'),
    (1, 'login', '2024-07-24 08:30:00'),
    (1, 'logout', '2024-07-24 12:30:00'),
    (2, 'login', '2024-07-24 09:30:00'),
    (2, 'logout', '2024-07-24 10:30:00')
]

# Create DataFrame
df_create = spark.createDataFrame(data, schema)
df=df_create.withColumn("activity_time", to_timestamp(col("activity_time"), "yyyy-MM-dd HH:mm:ss"))
# Show DataFrame
dshow(truncate=False)
dprintSchema()


+-----------+-------------+-------------------+
|employee_id|activity_type|activity_time      |
+-----------+-------------+-------------------+
|1          |login        |2024-07-23 08:00:00|
|1          |logout       |2024-07-23 12:00:00|
|1          |login        |2024-07-23 13:00:00|
|1          |logout       |2024-07-23 17:00:00|
|2          |login        |2024-07-23 09:00:00|
|2          |logout       |2024-07-23 11:00:00|
|2          |login        |2024-07-23 12:00:00|
|2          |logout       |2024-07-23 15:00:00|
|1          |login        |2024-07-24 08:30:00|
|1          |logout       |2024-07-24 12:30:00|
|2          |login        |2024-07-24 09:30:00|
|2          |logout       |2024-07-24 10:30:00|
+-----------+-------------+-------------------+

root
 |-- employee_id: integer (nullable = true)
 |-- activity_type: string (nullable = true)
 |-- activity_time: timestamp (nullable = true)



In [0]:
dwithColumn("login",lead(col("activity_time")).over(Window.partitionBy("employee_id").orderBy(col("activity_time"))))\
    .filter(col("activity_type")=='login')\
    .withColumn("activity_date",to_date(col("activity_time")))\
    .withColumnRenamed("activity_time","logout")\
    .withColumn("work_time",timestamp_diff('hour',col('logout'),col('login')))\
    .groupBy("employee_id","activity_date")\
    .agg(min(col("login")).alias("login"),max("logout").alias("logout"),sum(col("work_time"))).alias("work_time")\
    .show()


+-----------+-------------+-------------------+-------------------+--------------+
|employee_id|activity_date|              login|             logout|sum(work_time)|
+-----------+-------------+-------------------+-------------------+--------------+
|          1|   2024-07-23|2024-07-23 12:00:00|2024-07-23 13:00:00|             8|
|          1|   2024-07-24|2024-07-24 12:30:00|2024-07-24 08:30:00|             4|
|          2|   2024-07-23|2024-07-23 11:00:00|2024-07-23 12:00:00|             5|
|          2|   2024-07-24|2024-07-24 10:30:00|2024-07-24 09:30:00|             1|
+-----------+-------------+-------------------+-------------------+--------------+



### prob-66

In [0]:

from pyspark.sql import SparkSession
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.appName("CovidCases").getOrCreate()

# Define schema
schema = StructType([
    StructField("record_date", DateType(), True),
    StructField("cases_count", IntegerType(), True)
])

# Full data from SQL INSERT
data = [
    ("2021-01-01",66),("2021-01-02",41),("2021-01-03",54),("2021-01-04",68),("2021-01-05",16),("2021-01-06",90),("2021-01-07",34),("2021-01-08",84),("2021-01-09",71),("2021-01-10",14),("2021-01-11",48),("2021-01-12",72),("2021-01-13",55),
    ("2021-02-01",38),("2021-02-02",57),("2021-02-03",42),("2021-02-04",61),("2021-02-05",25),("2021-02-06",78),("2021-02-07",33),("2021-02-08",93),("2021-02-09",62),("2021-02-10",15),("2021-02-11",52),("2021-02-12",76),("2021-02-13",45),
    ("2021-03-01",27),("2021-03-02",47),("2021-03-03",36),("2021-03-04",64),("2021-03-05",29),("2021-03-06",81),("2021-03-07",32),("2021-03-08",89),("2021-03-09",63),("2021-03-10",19),("2021-03-11",53),("2021-03-12",78),("2021-03-13",49),
    ("2021-04-01",39),("2021-04-02",58),("2021-04-03",44),("2021-04-04",65),("2021-04-05",30),("2021-04-06",87),("2021-04-07",37),("2021-04-08",95),("2021-04-09",60),("2021-04-10",13),("2021-04-11",50),("2021-04-12",74),("2021-04-13",46),
    ("2021-05-01",28),("2021-05-02",49),("2021-05-03",35),("2021-05-04",67),("2021-05-05",26),("2021-05-06",82),("2021-05-07",31),("2021-05-08",92),("2021-05-09",61),("2021-05-10",18),("2021-05-11",54),("2021-05-12",79),("2021-05-13",51),
    ("2021-06-01",40),("2021-06-02",59),("2021-06-03",43),("2021-06-04",66),("2021-06-05",27),("2021-06-06",85),("2021-06-07",38),("2021-06-08",94),("2021-06-09",64),("2021-06-10",17),("2021-06-11",55),("2021-06-12",77),("2021-06-13",48),
    ("2021-07-01",34),("2021-07-02",50),("2021-07-03",37),("2021-07-04",69),("2021-07-05",32),("2021-07-06",80),("2021-07-07",33),("2021-07-08",88),("2021-07-09",57),("2021-07-10",21),("2021-07-11",56),("2021-07-12",73),("2021-07-13",42),
    ("2021-08-01",41),("2021-08-02",53),("2021-08-03",39),("2021-08-04",62),("2021-08-05",23),("2021-08-06",83),("2021-08-07",29),("2021-08-08",91),("2021-08-09",59),("2021-08-10",22),("2021-08-11",51),("2021-08-12",75),("2021-08-13",44),
    ("2021-09-01",36),("2021-09-02",45),("2021-09-03",40),("2021-09-04",68),("2021-09-05",28),("2021-09-06",84),("2021-09-07",30),("2021-09-08",90),("2021-09-09",61),("2021-09-10",20),("2021-09-11",52),("2021-09-12",71),("2021-09-13",43),
    ("2021-10-01",46),("2021-10-02",58),("2021-10-03",41),("2021-10-04",63),("2021-10-05",24),("2021-10-06",82),("2021-10-07",34),("2021-10-08",86),("2021-10-09",56),("2021-10-10",14),("2021-10-11",57),("2021-10-12",70),("2021-10-13",47),
    ("2021-11-01",31),("2021-11-02",44),("2021-11-03",38),("2021-11-04",67),("2021-11-05",22),("2021-11-06",79),("2021-11-07",32),("2021-11-08",94),("2021-11-09",60),("2021-11-10",15),("2021-11-11",54),("2021-11-12",73),("2021-11-13",46),
    ("2021-12-01",29),("2021-12-02",50),("2021-12-03",42),("2021-12-04",65),("2021-12-05",25),("2021-12-06",83),("2021-12-07",30),("2021-12-08",93),("2021-12-09",58),("2021-12-10",19),("2021-12-11",52),("2021-12-12",75),("2021-12-13",48)
]

# Convert string dates to datetime.date
data = [(datetime.strptime(d, "%Y-%m-%d").date(), c) for d, c in data]

# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

# Show DataFrame
dshow(10)  # Show first 10 rows
dprintSchema()

+-----------+-----------+
|record_date|cases_count|
+-----------+-----------+
| 2021-01-01|         66|
| 2021-01-02|         41|
| 2021-01-03|         54|
| 2021-01-04|         68|
| 2021-01-05|         16|
| 2021-01-06|         90|
| 2021-01-07|         34|
| 2021-01-08|         84|
| 2021-01-09|         71|
| 2021-01-10|         14|
+-----------+-----------+
only showing top 10 rows
root
 |-- record_date: date (nullable = true)
 |-- cases_count: integer (nullable = true)



In [0]:
df1=dwithColumn("record_month",month(col("record_date")))\
    .groupBy(col("record_month")).agg(sum(col("cases_count")).alias("cases_count"))
df12=df1.withColumn("cumulative_cases_prev_months",sum("cases_count").over(Window.orderBy("record_month").rowsBetween(Window.unboundedPreceding,-1))).fillna(0,subset=['cumulative_cases_prev_months']).show()

+------------+-----------+----------------------------+
|record_month|cases_count|cumulative_cases_prev_months|
+------------+-----------+----------------------------+
|           1|        713|                           0|
|           2|        677|                         713|
|           3|        667|                        1390|
|           4|        698|                        2057|
|           5|        673|                        2755|
|           6|        713|                        3428|
|           7|        672|                        4141|
|           8|        672|                        4813|
|           9|        668|                        5485|
|          10|        678|                        6153|
|          11|        655|                        6831|
|          12|        669|                        7486|
+------------+-----------+----------------------------+



In [0]:
df1_a = df1.alias("a")
df1_b = df1.alias("b")

df3 = (
    df1_a.join(
        df1_b,
        col("a.record_month") > col("b.record_month"),
        "left"
    )).orderBy(col("a.record_month")).groupBy("a.record_month","a.cases_count").agg(sum(col("b.cases_count")).alias("cummlative_cases_count")).orderBy("record_month")
df3.show()

+------------+-----------+----------------------+
|record_month|cases_count|cummlative_cases_count|
+------------+-----------+----------------------+
|           1|        713|                  NULL|
|           2|        677|                   713|
|           3|        667|                  1390|
|           4|        698|                  2057|
|           5|        673|                  2755|
|           6|        713|                  3428|
|           7|        672|                  4141|
|           8|        672|                  4813|
|           9|        668|                  5485|
|          10|        678|                  6153|
|          11|        655|                  6831|
|          12|        669|                  7486|
+------------+-----------+----------------------+



## prob-64

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.appName("StockData").getOrCreate()

# Define schema
schema = StructType([
    StructField("supplier_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("stock_quantity", IntegerType(), True),
    StructField("record_date", DateType(), True)
])

# Data from SQL INSERT
data = [
    (1, 1, 60, '2022-01-01'),
    (1, 1, 40, '2022-01-02'),
    (1, 1, 35, '2022-01-03'),
    (1, 1, 45, '2022-01-04'),
    (1, 1, 51, '2022-01-06'),
    (1, 1, 55, '2022-01-09'),
    (1, 1, 25, '2022-01-10'),
    (1, 1, 48, '2022-01-11'),
    (1, 1, 45, '2022-01-15'),
    (1, 1, 38, '2022-01-16'),
    (1, 2, 45, '2022-01-08'),
    (1, 2, 40, '2022-01-09'),
    (2, 1, 45, '2022-01-06'),
    (2, 1, 55, '2022-01-07'),
    (2, 2, 45, '2022-01-08'),
    (2, 2, 48, '2022-01-09'),
    (2, 2, 35, '2022-01-10'),
    (2, 2, 52, '2022-01-15'),
    (2, 2, 23, '2022-01-16')
]

# Convert date strings to datetime.date
data = [(s, p, q, datetime.strptime(d, "%Y-%m-%d").date()) for s, p, q, d in data]

# Create DataFrame
stock_df = spark.createDataFrame(data, schema=schema)

# Show DataFrame
stock_dshow()



+-----------+----------+--------------+-----------+
|supplier_id|product_id|stock_quantity|record_date|
+-----------+----------+--------------+-----------+
|          1|         1|            60| 2022-01-01|
|          1|         1|            40| 2022-01-02|
|          1|         1|            35| 2022-01-03|
|          1|         1|            45| 2022-01-04|
|          1|         1|            51| 2022-01-06|
|          1|         1|            55| 2022-01-09|
|          1|         1|            25| 2022-01-10|
|          1|         1|            48| 2022-01-11|
|          1|         1|            45| 2022-01-15|
|          1|         1|            38| 2022-01-16|
|          1|         2|            45| 2022-01-08|
|          1|         2|            40| 2022-01-09|
|          2|         1|            45| 2022-01-06|
|          2|         1|            55| 2022-01-07|
|          2|         2|            45| 2022-01-08|
|          2|         2|            48| 2022-01-09|
|          2

In [0]:
stock_dfilter('stock_quantity < 50').withColumn("g_flag",dateadd("record_date",-(row_number().over(Window.partitionBy("supplier_id","product_id").orderBy("record_date")))))\
    .groupBy("supplier_id","product_id","g_flag").agg(count("*").alias("cnt"),min("record_date").alias("record_date")).filter(col("cnt")>=2).drop("g_flag").show()

+-----------+----------+---+-----------+
|supplier_id|product_id|cnt|record_date|
+-----------+----------+---+-----------+
|          1|         1|  3| 2022-01-02|
|          1|         1|  2| 2022-01-10|
|          1|         1|  2| 2022-01-15|
|          1|         2|  2| 2022-01-08|
|          2|         2|  3| 2022-01-08|
+-----------+----------+---+-----------+



### prob-61

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.types import *

# Initialize Spark Session
spark = SparkSession.builder.appName("CustomersData").getOrCreate()

# Define schema
schema = StructType([
    StructField("customer_name", StringType(), True)
])

# Data from SQL INSERT
data = [
    ("Ankit Bansal",),
    ("Vishal Pratap Singh",),
    ("Michael",)
]

# Create DataFrame
customers_df = spark.createDataFrame(data, schema=schema)

# Show DataFrame
customers_df.show()


+-------------------+
|      customer_name|
+-------------------+
|       Ankit Bansal|
|Vishal Pratap Singh|
|            Michael|
+-------------------+



In [0]:
customers_df.withColumn("part",explode(split("customer_name"," " )))\
    .withColumn("rnk",row_number().over(Window.partitionBy("customer_name").orderBy("customer_name")))\
    .withColumn("cnt",count("*").over(Window.partitionBy("customer_name")))\
    .withColumn("first_name",when(col("rnk")==1,col("part")).otherwise("NONE"))\
    .withColumn("middle_name",when((col("rnk")==2) &  (col("cnt")==3),col("part")).otherwise("NONE"))\
    .withColumn("last_name",when(((col("rnk")==2) &(col("cnt")==2)) |  ((col("rnk")==3) & (col("cnt")==3)),col("part")).otherwise("NONE")).show()


+-------------------+-------+---+---+----------+-----------+---------+
|      customer_name|   part|rnk|cnt|first_name|middle_name|last_name|
+-------------------+-------+---+---+----------+-----------+---------+
|       Ankit Bansal|  Ankit|  1|  2|     Ankit|       NONE|     NONE|
|       Ankit Bansal| Bansal|  2|  2|      NONE|       NONE|   Bansal|
|            Michael|Michael|  1|  1|   Michael|       NONE|     NONE|
|Vishal Pratap Singh| Vishal|  1|  3|    Vishal|       NONE|     NONE|
|Vishal Pratap Singh| Pratap|  2|  3|      NONE|     Pratap|     NONE|
|Vishal Pratap Singh|  Singh|  3|  3|      NONE|       NONE|    Singh|
+-------------------+-------+---+---+----------+-----------+---------+



## prob-62


In [0]:

from pyspark.sql import SparkSession
from pyspark.sql import *
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimeType

# Initialize Spark Session
spark = SparkSession.builder.appName("UserInteractionsAnalysis").getOrCreate()

# Define schema
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("event", StringType(), True),
    StructField("event_date", DateType(), True),
    StructField("interaction_type", StringType(), True),
    StructField("game_id", StringType(), True),
    StructField("event_time", StringType(), True)  # We'll keep time as string for simplicity
])

# Data from SQL INSERT
data = [
    ("abc", "game_start", "2024-01-01", None, "ab0000", "10:00:00"),
    ("def", "game_start", "2024-01-01", None, "ab0000", "10:00:00"),
    ("def", "send_emoji", "2024-01-01", "emoji1", "ab0000", "10:03:20"),
    ("def", "send_message", "2024-01-01", "preloaded_quick", "ab0000", "10:03:49"),
    ("abc", "send_gift", "2024-01-01", "gift1", "ab0000", "10:04:40"),
    ("abc", "game_end", "2024-01-01", None, "ab0000", "10:10:00"),
    ("def", "game_end", "2024-01-01", None, "ab0000", "10:10:00"),
    ("abc", "game_start", "2024-01-01", None, "ab9999", "10:00:00"),
    ("def", "game_start", "2024-01-01", None, "ab9999", "10:00:00"),
    ("abc", "send_message", "2024-01-01", "custom_typed", "ab9999", "10:02:43"),
    ("abc", "send_gift", "2024-01-01", "gift1", "ab9999", "10:04:40"),
    ("abc", "game_end", "2024-01-01", None, "ab9999", "10:10:00"),
    ("def", "game_end", "2024-01-01", None, "ab9999", "10:10:00"),
    ("abc", "game_start", "2024-01-01", None, "ab1111", "10:00:00"),
    ("def", "game_start", "2024-01-01", None, "ab1111", "10:00:00"),
    ("abc", "game_end", "2024-01-01", None, "ab1111", "10:10:00"),
    ("def", "game_end", "2024-01-01", None, "ab1111", "10:10:00"),
    ("abc", "game_start", "2024-01-01", None, "ab1234", "10:00:00"),
    ("def", "game_start", "2024-01-01", None, "ab1234", "10:00:00"),
    ("abc", "send_message", "2024-01-01", "custom_typed", "ab1234", "10:02:43"),
    ("def", "send_emoji", "2024-01-01", "emoji1", "ab1234", "10:03:20"),
    ("def", "send_message", "2024-01-01", "preloaded_quick", "ab1234", "10:03:49"),
    ("abc", "send_gift", "2024-01-01", "gift1", "ab1234", "10:04:40"),
    ("abc", "game_end", "2024-01-01", None, "ab1234", "10:10:00"),
    ("def", "game_end", "2024-01-01", None, "ab1234", "10:10:00")
]

# Convert date strings to proper date type
from datetime import datetime
data = [(u, e, datetime.strptime(d, "%Y-%m-%d").date(), i, g, t) for u, e, d, i, g, t in data]

# Create DataFrame
user_interactions_df = spark.createDataFrame(data, schema=schema)

# Apply the logic equivalent to your SQL CASE WHEN
result_df = (
    user_interactions_df
    .groupBy("game_id")
    .agg(
        count("interaction_type").alias("interaction_count"),
        countDistinct(when(col("interaction_type").isNotNull(), col("user_id"))).alias("distinct_users"),
        countDistinct(when(col("interaction_type") == "custom_typed", col("user_id"))).alias("custom_users")
    )
    .withColumn(
        "game_type",
        when(col("interaction_count") == 0, "No Social Interaction")
         .when(col("distinct_users") == 1, "One Sided Interaction")
         .when(col("custom_users") >= 1, "Both sided with custom from one side")
         .otherwise("Both sided interaction without custom")
    )
    .select("game_id", "game_type")
).show(truncate=False)

# Show result


+-------+-------------------------------------+
|game_id|game_type                            |
+-------+-------------------------------------+
|ab0000 |Both sided interaction without custom|
|ab9999 |One Sided Interaction                |
|ab1111 |No Social Interaction                |
|ab1234 |Both sided with custom from one side |
+-------+-------------------------------------+



## recursive cte in **pyspark**
## __

In [0]:

from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.appName("RecursiveCTEEquivalent").getOrCreate()

# Sample DataFrame (job_positions)
data = [
    (1, "Engineer", "Level 1", 3),
    (2, "Manager", "Level 2", 2)
]
columns = ["id", "title", "payscale", "totalpost"]

df = spark.createDataFrame(data, columns)

# Generate sequence from 1 to totalpost for each row
df_with_sequence = df.withColumn("t", sequence(lit(1), col("totalpost")))

# Explode the sequence to create multiple rows
result_df = df_with_sequence.withColumn("t", explode(col("t")))

result_df.show()


df.withColumn(
    "t",
    sequence(col("totalpost"), lit(1), lit(-1))
).show()


+---+--------+--------+---------+---+
| id|   title|payscale|totalpost|  t|
+---+--------+--------+---------+---+
|  1|Engineer| Level 1|        3|  1|
|  1|Engineer| Level 1|        3|  2|
|  1|Engineer| Level 1|        3|  3|
|  2| Manager| Level 2|        2|  1|
|  2| Manager| Level 2|        2|  2|
+---+--------+--------+---------+---+

+---+--------+--------+---------+---------+
| id|   title|payscale|totalpost|        t|
+---+--------+--------+---------+---------+
|  1|Engineer| Level 1|        3|[3, 2, 1]|
|  2| Manager| Level 2|        2|   [2, 1]|
+---+--------+--------+---------+---------+



In [0]:
from pyspark.sql.functions import col, min as spark_min, max as spark_max, sequence, explode, lit, add_months, months_between, expr

# Sample DataFrame (SKU)
data = [
    ("2024-01-15",),
    ("2024-03-10",),
    ("2024-06-20",)
]
columns = ["PRICE_DATE"]

df = spark.createDataFrame(data, columns)

# Convert to date type
df = df.withColumn("PRICE_DATE", col("PRICE_DATE").cast("date"))

# Get min and max dates
agg_df = df.agg(
    spark_min("PRICE_DATE").alias("min_date"),
    spark_max("PRICE_DATE").alias("max_date")
)

# Calculate number of months difference
agg_df = agg_df.withColumn(
    "month_diff",
    months_between(col("max_date"), col("min_date")).cast("int")
)

# Generate sequence of dates by adding months
agg_df = agg_df.withColumn(
    "date_seq",
    sequence(
        col("min_date"),
        add_months(col("min_date"), col("month_diff")),
        expr("INTERVAL 1 MONTH")
    )
)

# Explode the sequence
result_df = agg_df.withColumn("dt", explode(col("date_seq"))).select("dt")

display(result_df)

dt
2024-01-15
2024-02-15
2024-03-15
2024-04-15
2024-05-15
2024-06-15


In [0]:

from pyspark.sql.functions import sequence, explode, col, lit

# Suppose min_date = 2024-01-01 and max_date = 2024-01-10
agg_df1 = agg_df.withColumn(
    "date_seq",
    sequence(col("min_date"), col("max_date"), expr("interval 1 month"))
)

# Explode to get individual dates
result_df1 = agg_df1.withColumn("dt", explode(col("date_seq"))).select("dt").show()


+----------+
|        dt|
+----------+
|2024-01-15|
|2024-02-15|
|2024-03-15|
|2024-04-15|
|2024-05-15|
|2024-06-15|
+----------+



In [0]:
from pyspark.sql.functions import sequence, explode, col, lit

# Suppose min_date = 2024-01-01 and max_date = 2024-01-10
agg_df1 = agg_df.withColumn(
    "date_seq",
    sequence(col("min_date"), col("max_date"), expr("interval 1 day"))
)

# Explode to get individual dates
result_df1 = agg_df1.withColumn("dt", explode(col("date_seq"))).select("dt").show()


+----------+
|        dt|
+----------+
|2024-01-15|
|2024-01-16|
|2024-01-17|
|2024-01-18|
|2024-01-19|
|2024-01-20|
|2024-01-21|
|2024-01-22|
|2024-01-23|
|2024-01-24|
|2024-01-25|
|2024-01-26|
|2024-01-27|
|2024-01-28|
|2024-01-29|
|2024-01-30|
|2024-01-31|
|2024-02-01|
|2024-02-02|
|2024-02-03|
+----------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import sequence, explode, col, lit

# Suppose min_date = 2024-01-01 and max_date = 2024-01-10
agg_df1 = agg_df.withColumn(
    "date_seq",
    sequence(col("max_date"), col("min_date"), expr("interval -1 month"))
)

# Explode to get individual dates
result_df1 = agg_df1.withColumn("dt", explode(col("date_seq"))).select("*").show(truncate=False)


+----------+----------+----------+------------------------------------------------------------------------+----------+
|min_date  |max_date  |month_diff|date_seq                                                                |dt        |
+----------+----------+----------+------------------------------------------------------------------------+----------+
|2024-01-15|2024-06-20|5         |[2024-06-20, 2024-05-20, 2024-04-20, 2024-03-20, 2024-02-20, 2024-01-20]|2024-06-20|
|2024-01-15|2024-06-20|5         |[2024-06-20, 2024-05-20, 2024-04-20, 2024-03-20, 2024-02-20, 2024-01-20]|2024-05-20|
|2024-01-15|2024-06-20|5         |[2024-06-20, 2024-05-20, 2024-04-20, 2024-03-20, 2024-02-20, 2024-01-20]|2024-04-20|
|2024-01-15|2024-06-20|5         |[2024-06-20, 2024-05-20, 2024-04-20, 2024-03-20, 2024-02-20, 2024-01-20]|2024-03-20|
|2024-01-15|2024-06-20|5         |[2024-06-20, 2024-05-20, 2024-04-20, 2024-03-20, 2024-02-20, 2024-01-20]|2024-02-20|
|2024-01-15|2024-06-20|5         |[2024-06-20, 2

In [0]:

from datetime import datetime
from pyspark.sql import Row

file_metadata = [
    ("s3://bucket/folder/file1.json", "file1.json", datetime(2026, 1, 4, 10, 30, 45)),
    ("s3://bucket/folder/file2.json", "file2.json", datetime(2026, 1, 4, 10, 32, 10)),
    ("s3://bucket/folder/file3.json", "file3.json", datetime(2026, 1, 4, 10, 40, 55))
]


In [0]:

metadata_rows = [
    Row(
        path=path,
        metadata_filename=filename,
        metadata_timestamp=lm.strftime("%Y-%m-%dT%H:%M:%S")
    )
    for path, filename, lm in file_metadata
]
print(metadata_rows)


[Row(path='s3://bucket/folder/file1.json', metadata_filename='file1.json', metadata_timestamp='2026-01-04T10:30:45'), Row(path='s3://bucket/folder/file2.json', metadata_filename='file2.json', metadata_timestamp='2026-01-04T10:32:10'), Row(path='s3://bucket/folder/file3.json', metadata_filename='file3.json', metadata_timestamp='2026-01-04T10:40:55')]


In [0]:

metadata_df = spark.createDataFrame(metadata_rows)
metadata_df.show(truncate=False)


+-----------------------------+-----------------+-------------------+
|path                         |metadata_filename|metadata_timestamp |
+-----------------------------+-----------------+-------------------+
|s3://bucket/folder/file1.json|file1.json       |2026-01-04T10:30:45|
|s3://bucket/folder/file2.json|file2.json       |2026-01-04T10:32:10|
|s3://bucket/folder/file3.json|file3.json       |2026-01-04T10:40:55|
+-----------------------------+-----------------+-------------------+

